# 시계열 미래 예측: lag·rolling·재귀 다중 스텝

각 Notebook은 독립 실행합니다. 기본값 DEMO=True는 합성 연습 데이터입니다. 실제 데이터는 설정 셀에서 DEMO=False와 경로·열·문제 유형을 지정하세요. 앞의 Notebook 실행이나 개인 모듈 설치가 필요하지 않습니다.

시간 예산은 모델 후보를 시작하기 전에 확인하는 소프트 제한입니다. 진행 중인 fit을 강제 중단하지 않습니다. 대회 지문과 공식 제출 규격을 우선합니다.

## 설정

단일/다중 개체의 등간격 단변량 예측입니다. 미래 외생변수와 불규칙 예측은 별도 설계가 필요합니다.

In [ ]:
DEMO=True
TRAIN_PATH='data/train.csv';TEST_PATH='data/test.csv'
TIME_COL='timestamp';ENTITY='series_id';VALUE='value';ID='id'
FREQUENCY='h' # pandas 주기. 등간격 예측에서 필수
LAGS=[1,2,3,24];ROLLING=[3,12,24]
VALID_FRACTION=.2;METRIC='rmse'
SAMPLE_PATH=None;TARGET_COLUMNS=['value'];OUTPUT='outputs/time_series/forecast.csv'


## 공통 함수

출력 ID와 제출 검사

In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, log_loss, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, classification_report,
    ConfusionMatrixDisplay, silhouette_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.dummy import DummyClassifier, DummyRegressor
SEED=42
rng=np.random.default_rng(SEED)

def read_table(path):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix.lower()=='.parquet': return pd.read_parquet(p)
    return pd.read_csv(p, sep='\t' if p.suffix.lower()=='.tsv' else ',')

def check_ids(df, id_col):
    if id_col not in df or df[id_col].isna().any() or df[id_col].duplicated().any():
        raise ValueError(f'{id_col}: 각 예측 단위에 결측 없는 고유 ID가 필요합니다.')

def split_rows(df, target=None, task='classification', strategy='random', group=None, time_col=None, fraction=.25, gap=0):
    idx=np.arange(len(df))
    if not 0<fraction<1: raise ValueError('validation fraction은 0~1 사이여야 합니다.')
    if strategy=='group':
        if not group or df[group].isna().any(): raise ValueError('유효한 그룹 열이 필요합니다.')
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=SEED).split(df,groups=df[group]))
        assert set(df.iloc[a][group]).isdisjoint(set(df.iloc[b][group]))
    elif strategy=='time':
        if not time_col: raise ValueError('시간 열을 지정하세요.')
        t=pd.to_datetime(df[time_col],errors='raise')
        if t.isna().any(): raise ValueError('시간 결측을 해결하세요.')
        unique=np.sort(t.unique());cut=int(len(unique)*(1-fraction))
        if cut<=gap or cut>=len(unique): raise ValueError('시간 분할에 필요한 데이터가 부족합니다.')
        a=idx[t<unique[cut-gap]];b=idx[t>=unique[cut]]
        assert t.iloc[a].max()<t.iloc[b].min()
    elif strategy in ('random','stratified'):
        strat=df[target] if task=='classification' and target else None
        if strat is not None and strat.value_counts().min()<2:
            raise ValueError('표본 1개인 클래스가 있습니다. 병합/수집/분할 정책을 검토하세요.')
        a,b=train_test_split(idx,test_size=fraction,random_state=SEED,stratify=strat)
    else: raise ValueError(f'지원하지 않는 분할: {strategy}')
    if not len(a) or not len(b): raise ValueError('빈 학습/검증 분할')
    if task=='classification' and target and set(df.iloc[b][target])-set(df.iloc[a][target]):
        raise ValueError('학습에 없는 클래스가 검증에 존재합니다.')
    return np.asarray(a),np.asarray(b)

def clean_tabular(X):
    out=X.copy()
    for c in out:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c]=pd.to_numeric(out[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        else: out[c]=out[c].map(lambda v: str(v) if pd.notna(v) else np.nan)
    return out

def tabular_preprocessor(X, robust=False):
    num=X.select_dtypes(include='number').columns.tolist()
    cat=[c for c in X if c not in num]
    parts=[]
    if num: parts.append(('num',make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RobustScaler() if robust else StandardScaler()),num))
    if cat: parts.append(('cat',make_pipeline(SimpleImputer(strategy='constant',fill_value='__MISSING__',keep_empty_features=True),OneHotEncoder(handle_unknown='ignore',min_frequency=2)),cat))
    if not parts: raise ValueError('특징 열이 없습니다.')
    return ColumnTransformer(parts)

def metrics_for(model,X,y,task):
    pred=model.predict(X)
    if task=='regression': return {'mae':float(mean_absolute_error(y,pred)),'rmse':float(np.sqrt(mean_squared_error(y,pred))),'r2':float(r2_score(y,pred))}
    out={'accuracy':float(accuracy_score(y,pred)),'f1_macro':float(f1_score(y,pred,average='macro',zero_division=0))}
    if hasattr(model,'predict_proba'):
        p=model.predict_proba(X);classes=model.classes_
        out['log_loss']=float(log_loss(y,p,labels=classes))
        if len(classes)==2 and len(np.unique(y))==2: out['roc_auc']=float(roc_auc_score(np.asarray(y)==classes[1],p[:,1]))
    return out

def fit_compare(candidates,X,y,a,b,task,metric,budget=120):
    direction={'accuracy':True,'f1_macro':True,'roc_auc':True,'r2':True,'log_loss':False,'mae':False,'rmse':False}
    if metric not in direction: raise ValueError('지원 지표를 선택하거나 metrics_for를 확장하세요.')
    t0=time.monotonic();rows=[];fitted={}
    for name,estimator in candidates.items():
        if rows and time.monotonic()-t0>=budget: break
        start=time.monotonic();model=clone(estimator).fit(X.iloc[a] if hasattr(X,'iloc') else X[a],y.iloc[a] if hasattr(y,'iloc') else y[a])
        stats=metrics_for(model,X.iloc[b] if hasattr(X,'iloc') else X[b],y.iloc[b] if hasattr(y,'iloc') else y[b],task)
        if metric not in stats or not np.isfinite(stats[metric]): raise ValueError(f'{metric}: 이 분할/모델에서 계산할 수 없습니다.')
        rows.append({'model':name,**stats,'seconds':time.monotonic()-start});fitted[name]=model
    table=pd.DataFrame(rows).sort_values(metric,ascending=not direction[metric]);display(table)
    best=table.iloc[0]['model']
    return best,fitted[best],table

def write_submission(ids,pred,id_col,target_cols,output,sample_path=None,probabilities=False):
    ids=pd.Series(ids).reset_index(drop=True)
    arr=np.asarray(pred)
    if arr.ndim==1: arr=arr[:,None]
    if arr.ndim!=2 or arr.shape!=(len(ids),len(target_cols)): raise ValueError('예측의 행/열 수와 제출 규격이 다릅니다.')
    if ids.isna().any() or ids.duplicated().any(): raise ValueError('제출 ID 결측/중복')
    if len(set(target_cols))!=len(target_cols) or id_col in target_cols: raise ValueError('제출 열 이름 중복')
    out=pd.DataFrame(arr,columns=target_cols);out.insert(0,id_col,ids.to_numpy())
    if out.isna().any().any(): raise ValueError('제출 값 결측')
    numeric=out[target_cols].select_dtypes(include='number')
    if numeric.size and not np.isfinite(numeric.to_numpy()).all(): raise ValueError('제출 값 무한대')
    if probabilities:
        v=arr.astype(float)
        if not np.isfinite(v).all() or (v<0).any() or (v>1).any(): raise ValueError('확률 범위 오류')
        if v.shape[1]>1 and not np.allclose(v.sum(axis=1),1,atol=1e-5): raise ValueError('확률 합 오류')
    if sample_path:
        sample=read_table(sample_path);check_ids(sample,id_col)
        if set(sample.columns)!=set(out.columns): raise ValueError('sample_submission과 열이 다릅니다.')
        if len(sample)!=len(out) or set(sample[id_col])!=set(out[id_col]): raise ValueError('sample_submission과 ID가 다릅니다. ID 자료형도 확인하세요.')
        out=sample[[id_col]].merge(out,on=id_col,how='left',validate='one_to_one')[sample.columns]
    p=Path(output);p.parent.mkdir(parents=True,exist_ok=True);out.to_csv(p,index=False)
    display(out.head());print('저장:',p,'형태:',out.shape)
    return out

def classification_output(model,X,kind='label',order=None,positive=None):
    if kind=='label': return model.predict(X)
    if not hasattr(model,'predict_proba'): raise ValueError('확률을 지원하는 모델이 필요합니다.')
    classes=list(model.classes_);p=model.predict_proba(X)
    if kind=='positive_probability':
        if positive not in classes: raise ValueError('positive_class를 실제 클래스 값으로 지정하세요.')
        return p[:,classes.index(positive)]
    if kind!='probability' or order is None or len(order)!=len(classes) or set(order)!=set(classes):
        raise ValueError('공식 제출 열에 대응하는 class_order를 지정하세요.')
    return p[:,[classes.index(v) for v in order]]


## 데이터

ENTITY=None이면 단일 시계열로 처리합니다. 테스트는 개체별 이력 직후부터 연속 미래 행이어야 합니다.

In [ ]:
if DEMO:
    records=[];future=[]
    for sid in ['A','B']:
        dates=pd.date_range('2025-01-01',periods=150,freq=FREQUENCY)
        values=5+np.sin(np.arange(150)*2*np.pi/24)+(sid=='B')*2+np.arange(150)*.01+rng.normal(0,.1,150)
        records.extend([{ENTITY:sid,TIME_COL:t,VALUE:v} for t,v in zip(dates,values)])
        future.extend([{ENTITY:sid,TIME_COL:t,ID:f'{sid}-{j}'} for j,t in enumerate(pd.date_range(dates[-1],periods=13,freq=FREQUENCY)[1:])])
    train=pd.DataFrame(records);test=pd.DataFrame(future)
else: train=read_table(TRAIN_PATH);test=read_table(TEST_PATH)
if not ENTITY:
    ENTITY='__series__';train[ENTITY]='single';test[ENTITY]='single'
for frame in [train,test]:
    frame[TIME_COL]=pd.to_datetime(frame[TIME_COL],errors='raise')
    if frame[[ENTITY,TIME_COL]].isna().any().any() or frame.duplicated([ENTITY,TIME_COL]).any(): raise ValueError('개체/시간 결측 또는 중복')
train[VALUE]=pd.to_numeric(train[VALUE],errors='raise')
if not np.isfinite(train[VALUE]).all(): raise ValueError('학습 값 결측/무한대를 먼저 해결하세요. EDA의 과거 기반 보간 예시 참고.')
train=train.sort_values([ENTITY,TIME_COL]).reset_index(drop=True)
check_ids(test,ID)
df=train


## 특징·시간 검증·재학습·제출

모든 개체에 공통 시간 경계를 적용합니다. 검증 horizon 전체를 재귀 예측하며 검증 정답을 lag에 넣지 않습니다. 평균/표준편차도 과거 값만 사용합니다.

In [ ]:
if not LAGS or min(LAGS)<1 or any(w<1 for w in ROLLING): raise ValueError('양의 lag/window 필요')
MAX_HISTORY=max(LAGS+ROLLING)

def regular_check(g):
    expected=pd.date_range(g[TIME_COL].iloc[0],periods=len(g),freq=FREQUENCY)
    if not np.array_equal(g[TIME_COL].to_numpy(),expected.to_numpy()): raise ValueError('불규칙 간격: 먼저 EDA에서 리샘플링 정책을 정하세요.')

def point_features(history,t,sid):
    if len(history)<MAX_HISTORY: raise ValueError('lag/rolling에 필요한 이력이 부족합니다.')
    row={ENTITY:str(sid),'hour_sin':np.sin(2*np.pi*t.hour/24),'hour_cos':np.cos(2*np.pi*t.hour/24),'weekday':t.dayofweek}
    for lag in LAGS: row[f'lag_{lag}']=history[-lag]
    for window in ROLLING:
        past=np.asarray(history[-window:]);row[f'mean_{window}']=past.mean();row[f'std_{window}']=past.std()
    return row

def training_matrix(frame):
    rows=[];labels=[]
    for sid,g in frame.groupby(ENTITY,sort=False):
        g=g.sort_values(TIME_COL);regular_check(g);values=g[VALUE].tolist()
        for j in range(MAX_HISTORY,len(g)):
            rows.append(point_features(values[:j],g[TIME_COL].iloc[j],sid));labels.append(values[j])
    if not rows: raise ValueError('학습 표본 부족: lag를 줄이거나 데이터를 늘리세요.')
    return pd.DataFrame(rows),np.asarray(labels)

def recursive_predict(model,history,future):
    answers=pd.Series(index=future.index,dtype=float)
    for sid,g in future.groupby(ENTITY,sort=False):
        g=g.sort_values(TIME_COL);past=history[history[ENTITY]==sid].sort_values(TIME_COL)
        if past.empty: raise ValueError(f'학습 이력이 없는 개체: {sid}')
        regular_check(past)
        expected=pd.date_range(past[TIME_COL].iloc[-1],periods=len(g)+1,freq=FREQUENCY)[1:]
        if not np.array_equal(g[TIME_COL].to_numpy(),expected.to_numpy()): raise ValueError('미래 행은 이력 직후부터 연속이어야 합니다.')
        values=past[VALUE].tolist()
        for idx,row in g.iterrows():
            value=float(model.predict(pd.DataFrame([point_features(values,row[TIME_COL],sid)]))[0])
            answers.loc[idx]=value;values.append(value) # 미래 정답이 아니라 이전 예측을 사용
    return answers.loc[future.index].to_numpy()

unique=np.sort(train[TIME_COL].unique());cut=int(len(unique)*(1-VALID_FRACTION))
if not 0<cut<len(unique): raise ValueError('검증 경계 오류')
boundary=unique[cut]
history=train[train[TIME_COL]<boundary].copy();valid=train[train[TIME_COL]>=boundary].copy()
X,y=training_matrix(history)
models={'ridge':make_pipeline(tabular_preprocessor(X),Ridge(alpha=10,solver='lsqr')),
        'trees':make_pipeline(tabular_preprocessor(X),ExtraTreesRegressor(n_estimators=60,min_samples_leaf=2,n_jobs=1,random_state=SEED))}
rows=[]
for name,estimator in models.items():
    estimator.fit(X,y);p=recursive_predict(estimator,history,valid)
    rows.append({'model':name,'rmse':float(np.sqrt(mean_squared_error(valid[VALUE],p))),'mae':float(mean_absolute_error(valid[VALUE],p))})
# 마지막 관측값을 유지하는 기준 예측
last=history.groupby(ENTITY)[VALUE].last();naive=valid[ENTITY].map(last)
rows.append({'model':'last_value','rmse':float(np.sqrt(mean_squared_error(valid[VALUE],naive))),'mae':float(mean_absolute_error(valid[VALUE],naive))})
if METRIC not in ('rmse','mae'): raise ValueError('예측 지표는 rmse 또는 mae를 선택하세요.')
results=pd.DataFrame(rows).sort_values(METRIC);display(results);best=results.iloc[0]['model']
# 단순 기준 모델이 선택되더라도 미래 개체/간격 계약을 검사
for sid,g in test.groupby(ENTITY,sort=False):
    past=train[train[ENTITY]==sid].sort_values(TIME_COL)
    if past.empty: raise ValueError('테스트에 학습 이력이 없는 개체가 있습니다.')
    regular_check(past)
    expected=pd.date_range(past[TIME_COL].iloc[-1],periods=len(g)+1,freq=FREQUENCY)[1:]
    if not np.array_equal(g.sort_values(TIME_COL)[TIME_COL].to_numpy(),expected.to_numpy()): raise ValueError('미래 시각이 이력 직후 연속 주기와 다릅니다.')
if best=='last_value': predictions=test[ENTITY].map(train.groupby(ENTITY)[VALUE].last()).to_numpy()
else:
    Xall,yall=training_matrix(train);final_model=clone(models[best]).fit(Xall,yall)
    predictions=recursive_predict(final_model,train,test)
submission=write_submission(test[ID],predictions,ID,TARGET_COLUMNS,OUTPUT,SAMPLE_PATH)
